# RunPod: Qwen3.8-27B Q4 with Ollama

This notebook uses the official `runpodctl` CLI to inspect the account, discover GPU capacity, create an on-demand RunPod GPU Pod, diagnose it, review billing, and delete it. Inside the Pod it installs Ollama, pulls `qwen3.8:27b-q4_K_M`, calls the model through a private SSH tunnel, and measures tokens per second.

The default is a 32,768-token context on one 32 GB RTX 5090. A 48 GB A40 or L40S is the recommended fallback if the 5090 is unavailable or runs out of memory.

> **Authentication:** the RunPod API key protects management operations performed by `runpodctl`; it does not protect a publicly exposed Ollama port. Ollama stays bound to remote `127.0.0.1:11434`, only SSH is exposed, and the local tunnel also binds to `127.0.0.1`.
>
> **Cost:** Pod creation starts billing. Creation and deletion require explicit Boolean switches, and the create command includes a four-hour automatic-termination backstop by default.
>
> **Status:** this workflow follows current official RunPod and Ollama documentation, but has not been executed against a paid RunPod account in this repository. Verify every paid action in the RunPod console.

## 1. Install and authenticate `runpodctl`

Install the official CLI outside the Python environment:

```bash
brew install runpod/runpodctl/runpodctl
runpodctl doctor
runpodctl version
runpodctl user
```

`runpodctl doctor` is RunPod's recommended interactive setup for the API key and SSH key. The credential is stored under `~/.runpod/config.toml`; protect it and never commit or print it.

From `02_Example_Deployments/lab`, install the locked Python environment and select its kernel:

```bash
uv sync --locked
uv run python -m ipykernel install --user --name gpu-deployments --display-name gpu-deployments
```

Required in the untracked `.env` file:

```dotenv
RUNPOD_SSH_KEY_PATH=/absolute/path/to/id_runpod_ed25519
```

The matching `.pub` file is passed to the official RunPod image. Only the public key is sent to the Pod.

In [ ]:
import json
import os
import re
import shlex
import shutil
import socket
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

RUNPODCTL_PATH = shutil.which("runpodctl")
SSH_KEY_PATH = Path(os.path.expanduser(
    os.getenv("RUNPOD_SSH_KEY_PATH", "~/.ssh/id_runpod_ed25519")
))
SSH_PUBLIC_KEY_PATH = Path(os.path.expanduser(
    os.getenv("RUNPOD_SSH_PUBLIC_KEY_PATH", f"{SSH_KEY_PATH}.pub")
))
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "").strip() or "qwen3.8:27b-q4_K_M"
OLLAMA_CONTEXT_LENGTH = int(os.getenv("OLLAMA_CONTEXT_LENGTH", "32768"))

if not RUNPODCTL_PATH:
    raise RuntimeError(
        "runpodctl is not installed. On macOS: brew install runpod/runpodctl/runpodctl"
    )
if not SSH_KEY_PATH.is_file():
    raise RuntimeError(f"SSH private key not found: {SSH_KEY_PATH}")
if not SSH_PUBLIC_KEY_PATH.is_file():
    raise RuntimeError(f"SSH public key not found: {SSH_PUBLIC_KEY_PATH}")
if OLLAMA_CONTEXT_LENGTH < 1024:
    raise ValueError("OLLAMA_CONTEXT_LENGTH should be at least 1024")

SSH_PUBLIC_KEY = SSH_PUBLIC_KEY_PATH.read_text().strip()
if not SSH_PUBLIC_KEY.startswith(("ssh-ed25519 ", "ssh-rsa ", "ecdsa-sha2-")):
    raise RuntimeError("The public-key file does not contain a recognizable SSH public key")

POD_ID = os.getenv("RUNPOD_POD_ID", "").strip() or None
TUNNEL = None
print(f"runpodctl: {RUNPODCTL_PATH}")
print(f"Model: {OLLAMA_MODEL}")
print(f"Context limit: {OLLAMA_CONTEXT_LENGTH:,} tokens")
print(f"SSH key: {SSH_KEY_PATH}")
print(f"Resume Pod: {POD_ID or '<none>'}")

In [ ]:
def runpodctl(args: list[str], *, expect_json: bool = True, timeout_s: int = 120):
    """Run the official CLI without printing credentials or the injected public key."""
    command = [RUNPODCTL_PATH, *args]
    display = command.copy()
    if "--env" in display:
        env_index = display.index("--env") + 1
        if env_index < len(display):
            display[env_index] = "<redacted environment JSON>"
    print("$", shlex.join(display))
    result = subprocess.run(
        command, capture_output=True, text=True, timeout=timeout_s
    )
    if result.returncode != 0:
        raise RuntimeError((result.stderr or result.stdout).strip())
    output = result.stdout.strip()
    if not expect_json:
        return output
    try:
        return json.loads(output) if output else None
    except json.JSONDecodeError as exc:
        raise RuntimeError(
            f"runpodctl did not return its expected JSON output: {output[:1000]}"
        ) from exc


def as_records(value) -> list[dict]:
    if isinstance(value, list):
        return [row for row in value if isinstance(row, dict)]
    if isinstance(value, dict):
        for key in ("items", "results", "data", "gpus", "pods"):
            nested = value.get(key)
            if isinstance(nested, list):
                return [row for row in nested if isinstance(row, dict)]
    return []


def find_pod_id(value) -> str | None:
    if isinstance(value, dict):
        for key in ("id", "podId"):
            candidate = value.get(key)
            if isinstance(candidate, str) and candidate:
                return candidate
        for nested in value.values():
            candidate = find_pod_id(nested)
            if candidate:
                return candidate
    elif isinstance(value, list):
        for nested in value:
            candidate = find_pod_id(nested)
            if candidate:
                return candidate
    elif isinstance(value, str) and re.fullmatch(r"[A-Za-z0-9_-]{6,}", value):
        return value
    return None


version = runpodctl(["version"], expect_json=False)
account = runpodctl(["user"])
print(version)
print("RunPod authentication verified.")
print(f"Balance: ${float(account.get('clientBalance', 0)):.2f}")
print(f"Current spend: ${float(account.get('currentSpendPerHr', 0)):.4f}/h")
print(f"Account spend limit: ${float(account.get('spendLimit', 0)):.2f}/h")

## 2. Review capacity and the Pod command

The notebook provisions with `runpodctl pod create`. It uses RunPod's official PyTorch image because that image supports SSH, then installs Ollama with Ollama's official installer.

Only `22/tcp` is exposed. The model cache lives at `/workspace/ollama-models`. The CLI accepts one GPU type per creation attempt, so the default is the 32 GB RTX 5090; change `RUNPOD_GPU_TYPE` to `NVIDIA A40` or `NVIDIA L40S` if necessary.

The automatic termination deadline is a cost backstop. Keep it enabled even though the notebook also has an explicit cleanup cell.

In [ ]:
RUNPOD_IMAGE = os.getenv("RUNPOD_IMAGE", "").strip() or "runpod/pytorch:latest"
RUNPOD_CLOUD_TYPE = os.getenv("RUNPOD_CLOUD_TYPE", "COMMUNITY").strip().upper()
RUNPOD_GPU_TYPE = os.getenv(
    "RUNPOD_GPU_TYPE", "NVIDIA GeForce RTX 5090"
).strip()
RUNPOD_GPU_COUNT = int(os.getenv("RUNPOD_GPU_COUNT", "1"))
RUNPOD_CONTAINER_DISK_GB = int(os.getenv("RUNPOD_CONTAINER_DISK_GB", "50"))
RUNPOD_VOLUME_GB = int(os.getenv("RUNPOD_VOLUME_GB", "40"))
RUNPOD_POD_NAME = os.getenv("RUNPOD_POD_NAME", "ollama-qwen3-8-27b-q4").strip()
RUNPOD_TERMINATE_AFTER = os.getenv("RUNPOD_TERMINATE_AFTER", "4h").strip()

if RUNPOD_CLOUD_TYPE not in {"COMMUNITY", "SECURE"}:
    raise ValueError("RUNPOD_CLOUD_TYPE must be COMMUNITY or SECURE")
if RUNPOD_GPU_COUNT < 1:
    raise ValueError("RUNPOD_GPU_COUNT must be at least 1")
if not RUNPOD_GPU_TYPE:
    raise ValueError("RUNPOD_GPU_TYPE cannot be empty")
if not RUNPOD_TERMINATE_AFTER:
    raise ValueError("Keep RUNPOD_TERMINATE_AFTER configured as a cost backstop")

available_gpus = runpodctl(["gpu", "list"])
matching_gpus = [
    row for row in as_records(available_gpus)
    if row.get("gpuId") == RUNPOD_GPU_TYPE or row.get("id") == RUNPOD_GPU_TYPE
]
print("Requested GPU:", RUNPOD_GPU_TYPE)
print("Current GPU record:", json.dumps(matching_gpus, indent=2)[:1600])
print("Fallbacks for 32K: NVIDIA A40 or NVIDIA L40S")

create_args = [
    "pod", "create",
    "--name", RUNPOD_POD_NAME,
    "--image", RUNPOD_IMAGE,
    "--gpu-id", RUNPOD_GPU_TYPE,
    "--gpu-count", str(RUNPOD_GPU_COUNT),
    "--cloud-type", RUNPOD_CLOUD_TYPE,
    "--ssh",
    "--ports", "22/tcp",
    "--container-disk-in-gb", str(RUNPOD_CONTAINER_DISK_GB),
    "--volume-in-gb", str(RUNPOD_VOLUME_GB),
    "--volume-mount-path", "/workspace",
    "--terminate-after", RUNPOD_TERMINATE_AFTER,
    "--env", json.dumps({"PUBLIC_KEY": SSH_PUBLIC_KEY}),
]
if RUNPOD_CLOUD_TYPE == "COMMUNITY":
    create_args.append("--public-ip")

print("Provisioning command prepared. The public-key environment value is hidden in logs.")
print("Automatic termination:", RUNPOD_TERMINATE_AFTER)

## 3. Create or resume the Pod

Creating a Pod starts billing. Run `runpodctl user`, inspect the configuration above, and check the Billing page first. Set `CREATE_POD = True` only when ready.

The CLI's JSON response supplies the Pod ID. If `RUNPOD_POD_ID` was loaded from `.env`, the cell uses `runpodctl pod get` to resume it instead of creating another Pod.

In [ ]:
CREATE_POD = False

if POD_ID:
    pod = runpodctl(["pod", "get", POD_ID])
    print(f"Resuming Pod {POD_ID}")
elif not CREATE_POD:
    print("Creation disabled. Review the configuration, then set CREATE_POD = True.")
else:
    created = runpodctl(create_args, timeout_s=300)
    POD_ID = find_pod_id(created)
    if not POD_ID:
        raise RuntimeError(f"Could not find the Pod ID in runpodctl output: {created}")
    pod = runpodctl(["pod", "get", POD_ID])
    print(f"Created Pod: {POD_ID}")
    print("Automatic termination deadline:", RUNPOD_TERMINATE_AFTER)
    print("Save RUNPOD_POD_ID in .env if you want to resume after restarting the kernel.")

### Cost snapshot and emergency deletion

`runpodctl user` shows balance and current total spend per hour. Pod billing history can be filtered to this Pod. Billing data may lag immediately after creation.

The deletion helper permanently deletes the Pod and its attached volume disk. Use it immediately if a later cell fails. Separately created network volumes are not deleted.

In [ ]:
account = runpodctl(["user"])
print(f"Balance: ${float(account.get('clientBalance', 0)):.2f}")
print(f"Current total spend: ${float(account.get('currentSpendPerHr', 0)):.4f}/h")

if POD_ID:
    billing = runpodctl([
        "billing", "pods",
        "--bucket-size", "hour",
        "--grouping", "podId",
        "--pod-id", POD_ID,
    ])
    print("Billing history for this Pod:")
    print(json.dumps(billing, indent=2)[:4000])
else:
    print("No Pod ID yet; per-Pod billing is not available.")


def delete_pod(pod_id: str) -> None:
    output = runpodctl(["pod", "delete", pod_id], expect_json=False)
    if output:
        print(output)


print("Emergency cleanup: delete_pod(POD_ID)")

## 4. Wait for SSH and inspect connection details

`runpodctl pod get` provides current resource details, while `runpodctl ssh info` provides the current connection command. The cell polls until a public IP and mapped port 22 exist, then verifies key-based SSH.

Host keys are accepted only on first use and stored in your normal SSH known-hosts file. If a recycled address later produces a mismatch, verify it in the authenticated RunPod console before changing known hosts.

In [ ]:
if not POD_ID:
    raise RuntimeError("Create a Pod or set RUNPOD_POD_ID before continuing")


def wait_for_ssh_details(pod_id: str, timeout_s: int = 900) -> tuple[dict, str, int]:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        current = runpodctl(["pod", "get", pod_id])
        public_ip = current.get("publicIp")
        mappings = current.get("portMappings") or {}
        ssh_port = mappings.get("22") or mappings.get(22)
        if public_ip and ssh_port:
            return current, public_ip, int(ssh_port)
        print("Waiting for public IP and SSH mapping...")
        time.sleep(10)
    raise TimeoutError("RunPod did not provide a public IP and mapped SSH port in time")


pod, SSH_HOST, SSH_PORT = wait_for_ssh_details(POD_ID)
ssh_info = runpodctl(["ssh", "info", POD_ID])
print("runpodctl SSH info:", json.dumps(ssh_info, indent=2))

SSH_PREFIX = [
    "ssh",
    "-i", str(SSH_KEY_PATH),
    "-p", str(SSH_PORT),
    "-o", "IdentitiesOnly=yes",
    "-o", "StrictHostKeyChecking=accept-new",
    "-o", "ConnectTimeout=10",
    f"root@{SSH_HOST}",
]


def wait_for_ssh(timeout_s: int = 600) -> None:
    deadline = time.time() + timeout_s
    last_error = "not attempted"
    while time.time() < deadline:
        result = subprocess.run(
            [*SSH_PREFIX, "true"], capture_output=True, text=True, timeout=20
        )
        if result.returncode == 0:
            print("SSH is ready.")
            return
        last_error = (result.stderr or result.stdout).strip()
        time.sleep(10)
    raise TimeoutError(f"SSH did not become ready: {last_error}")


print(f"Pod: {POD_ID}")
print(f"GPU: {(pod.get('gpu') or {}).get('displayName', '<pending>')}")
print(f"SSH: root@{SSH_HOST}:{SSH_PORT}")
wait_for_ssh()

## 5. Install and start Ollama

This follows RunPod's Ollama tutorial by installing `curl`, `lshw`, and `zstd`, then running Ollama's official installation script. The service is restarted with:

- `OLLAMA_HOST=127.0.0.1:11434`
- `OLLAMA_MODELS=/workspace/ollama-models`
- `OLLAMA_CONTEXT_LENGTH=32768` by default

The service log is stored at `/workspace/ollama-serve.log`. Review the remote install URL before running this cell in a high-trust environment.

In [ ]:
def ssh_run(script: str, *, timeout_s: int = 600) -> str:
    command = [*SSH_PREFIX, "bash", "-lc", shlex.quote(script)]
    result = subprocess.run(
        command, capture_output=True, text=True, timeout=timeout_s
    )
    if result.returncode != 0:
        raise RuntimeError((result.stderr or result.stdout).strip())
    return result.stdout.strip()


remote_setup = f"""
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive
apt-get update
apt-get install -y ca-certificates curl lshw zstd
if ! command -v ollama >/dev/null 2>&1; then
    curl -fsSL https://ollama.com/install.sh | sh
fi
mkdir -p /workspace/ollama-models
pkill -x ollama 2>/dev/null || true
nohup env \\
    OLLAMA_HOST=127.0.0.1:11434 \\
    OLLAMA_MODELS=/workspace/ollama-models \\
    OLLAMA_CONTEXT_LENGTH={OLLAMA_CONTEXT_LENGTH} \\
    ollama serve >/workspace/ollama-serve.log 2>&1 </dev/null &
sleep 3
curl -fsS http://127.0.0.1:11434/api/tags >/dev/null
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
"""
print(ssh_run(remote_setup, timeout_s=900))
print("Ollama is running on remote localhost only.")

## 6. Open the private SSH tunnel

The notebook chooses an unused local port and forwards it to Ollama's remote localhost socket. `ExitOnForwardFailure` makes SSH fail immediately if the forwarding cannot be established.

In [ ]:
def free_local_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("127.0.0.1", 0))
        return int(sock.getsockname()[1])


def wait_for_local_port(port: int, timeout_s: int = 30) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return
        if TUNNEL is not None and TUNNEL.poll() is not None:
            stderr = TUNNEL.stderr.read() if TUNNEL.stderr else ""
            raise RuntimeError(f"SSH tunnel exited early: {stderr}")
        time.sleep(0.5)
    raise TimeoutError("The local SSH tunnel did not open in time")


if TUNNEL is not None and TUNNEL.poll() is None:
    TUNNEL.terminate()
    TUNNEL.wait(timeout=10)

LOCAL_PORT = free_local_port()
TUNNEL = subprocess.Popen(
    [
        "ssh",
        "-i", str(SSH_KEY_PATH),
        "-p", str(SSH_PORT),
        "-o", "IdentitiesOnly=yes",
        "-o", "StrictHostKeyChecking=accept-new",
        "-o", "ExitOnForwardFailure=yes",
        "-o", "ServerAliveInterval=30",
        "-N",
        "-L", f"127.0.0.1:{LOCAL_PORT}:127.0.0.1:11434",
        f"root@{SSH_HOST}",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True,
)
wait_for_local_port(LOCAL_PORT)
OLLAMA_URL = f"http://127.0.0.1:{LOCAL_PORT}"
print(f"Private Ollama URL: {OLLAMA_URL}")

## 7. Pull Qwen3.8-27B Q4_K_M

Ollama's `POST /api/pull` endpoint streams newline-delimited JSON. The helper reports progress and also catches errors delivered after the HTTP stream begins. The approximately 18 GB download can take several minutes.

In [ ]:
def wait_for_ollama(base_url: str, timeout_s: int = 180) -> None:
    deadline = time.time() + timeout_s
    last_error = "not reachable"
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(f"{base_url}/api/tags", timeout=10) as response:
                json.load(response)
            print("Ollama is ready through the SSH tunnel.")
            return
        except Exception as exc:
            last_error = str(exc)
            time.sleep(5)
    raise TimeoutError(f"Ollama did not become ready: {last_error}")


def pull_model(base_url: str, model: str) -> None:
    body = json.dumps({"model": model, "stream": True}).encode()
    request = urllib.request.Request(
        f"{base_url}/api/pull",
        data=body,
        headers={"Content-Type": "application/json"},
    )
    last_status = None
    last_bucket = -1
    try:
        with urllib.request.urlopen(request, timeout=1800) as response:
            for raw_line in response:
                event = json.loads(raw_line)
                if event.get("error"):
                    raise RuntimeError(event["error"])
                status = event.get("status", "working")
                total = event.get("total") or 0
                completed = event.get("completed") or 0
                percent = int(completed * 100 / total) if total else None
                bucket = percent // 5 if percent is not None else -1
                if status != last_status or bucket != last_bucket:
                    suffix = f" {percent}%" if percent is not None else ""
                    print(f"{status}{suffix}")
                    last_status, last_bucket = status, bucket
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode(errors="replace")
        raise RuntimeError(f"Ollama pull failed ({exc.code}): {detail}") from exc


def show_model(base_url: str, model: str) -> dict:
    request = urllib.request.Request(
        f"{base_url}/api/show",
        data=json.dumps({"model": model, "verbose": False}).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)


wait_for_ollama(OLLAMA_URL)
pull_model(OLLAMA_URL, OLLAMA_MODEL)
model_info = show_model(OLLAMA_URL, OLLAMA_MODEL)
print("Pulled:", model_info.get("details", {}))

## 8. Test the OpenAI-compatible endpoint

This coding problem requires an in-place linear-time algorithm and gives enough edge cases to reveal whether the model understands the constraints. `max_tokens` is deliberately omitted, so there is no application-level output cap. `reasoning_effort="none"` disables extended thinking for a faster test.

In [ ]:
PROMPT = """
Write a Python function:

```python
def smallest_missing_positive(numbers: list[int]) -> int:
```

Given an unsorted list of integers, return the smallest positive integer that does not occur in the list.

Requirements:

- Run in `O(n)` time.
- Use `O(1)` additional space.
- Modify the input list if necessary.
- Do not use `set()`, sorting, or additional lists.
- Explain the algorithm briefly.
- Include tests for empty input, duplicates, negative numbers, and already consecutive values.

Examples:

```python
smallest_missing_positive([3, 4, -1, 1]) == 2
smallest_missing_positive([1, 2, 0]) == 3
smallest_missing_positive([7, 8, 9, 11, 12]) == 1
smallest_missing_positive([1, 1, 2, 2]) == 3
```
"""

In [ ]:
client = OpenAI(base_url=f"{OLLAMA_URL}/v1/", api_key="ollama")

started = time.perf_counter()
response = client.chat.completions.create(
    model=OLLAMA_MODEL,
    messages=[{"role": "user", "content": PROMPT}],
    reasoning_effort="none",
)
elapsed = time.perf_counter() - started

print(response.choices[0].message.content)
print(f"\n\nLatency including first model load: {elapsed:.1f}s")

with urllib.request.urlopen(f"{OLLAMA_URL}/api/ps", timeout=10) as result:
    running = json.load(result).get("models", [])
print("Running model details:", json.dumps(running, indent=2)[:1600])

### 8.1 Measure tokens per second

The OpenAI-compatible measurement is user-visible end-to-end throughput, including tunnel overhead and model loading. Ollama's native `eval_count / eval_duration` is generation-only throughput and is better for comparing GPUs. Durations are reported in nanoseconds.

The native call is a second, warm request so initial loading has already occurred.

In [ ]:
if response.usage is None or response.usage.completion_tokens is None:
    print("The OpenAI-compatible response did not include token usage.")
else:
    output_tokens = response.usage.completion_tokens
    end_to_end_tok_s = output_tokens / elapsed
    print(f"OpenAI-compatible output tokens: {output_tokens}")
    print(f"End-to-end throughput: {end_to_end_tok_s:.2f} tok/s")

payload = {
    "model": OLLAMA_MODEL,
    "messages": [{"role": "user", "content": PROMPT}],
    "stream": False,
    "think": False,
}
request = urllib.request.Request(
    f"{OLLAMA_URL}/api/chat",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)

with urllib.request.urlopen(request, timeout=900) as result:
    metrics = json.load(result)

eval_count = metrics.get("eval_count", 0)
eval_duration_s = metrics.get("eval_duration", 0) / 1e9
generation_tok_s = eval_count / eval_duration_s if eval_duration_s else 0.0

prompt_count = metrics.get("prompt_eval_count", 0)
cached_count = metrics.get("prompt_eval_cached_count", 0)
uncached_prompt_count = max(prompt_count - cached_count, 0)
prompt_duration_s = metrics.get("prompt_eval_duration", 0) / 1e9
prompt_tok_s = uncached_prompt_count / prompt_duration_s if prompt_duration_s else 0.0

print("\nWarm native Ollama response:\n")
print(metrics["message"]["content"])
print(f"\nGenerated tokens: {eval_count}")
print(f"Generation speed: {generation_tok_s:.2f} tok/s")
print(f"Uncached prompt speed: {prompt_tok_s:.2f} tok/s")
print(f"Model load: {metrics.get('load_duration', 0) / 1e9:.2f}s")
print(f"Total native request: {metrics.get('total_duration', 0) / 1e9:.2f}s")

## 9. Inspect and diagnose the Pod

Use `runpodctl pod get` for control-plane status, then inspect GPU memory, the loaded Ollama model, storage, and the service log over authenticated SSH.

In [ ]:
pod_status = runpodctl(["pod", "get", POD_ID])
print(json.dumps(pod_status, indent=2)[:4000])

inspection = ssh_run(
    "nvidia-smi; echo; OLLAMA_HOST=127.0.0.1:11434 ollama ps; "
    "echo; df -h /workspace; echo; tail -n 40 /workspace/ollama-serve.log",
    timeout_s=120,
)
print(inspection)

billing = runpodctl([
    "billing", "pods",
    "--bucket-size", "hour",
    "--grouping", "podId",
    "--pod-id", POD_ID,
])
print("\nPod billing history:\n", json.dumps(billing, indent=2)[:4000])

## 10. Cleanup and verify costs

Closing the SSH tunnel does not stop billing. `runpodctl pod stop POD_ID` releases GPU compute but retains the Pod and its volume disk, so storage charges can continue.

Set `DELETE_POD = True` to run `runpodctl pod delete POD_ID`, which permanently deletes the Pod and its attached volume disk. The cell then lists all Pods and prints the account spend snapshot. A separately created network volume must be deleted independently.

The creation command's `--terminate-after` deadline is a fallback, not a substitute for explicit cleanup.

In [ ]:
DELETE_POD = False

if TUNNEL is not None and TUNNEL.poll() is None:
    TUNNEL.terminate()
    TUNNEL.wait(timeout=10)
    print("SSH tunnel closed.")

if not POD_ID:
    print("No Pod ID is configured.")
elif not DELETE_POD:
    print(f"Pod {POD_ID} is still present and may still be billing.")
    print("Set DELETE_POD = True and rerun this cell when finished.")
    print(f"Automatic termination remains configured for: {RUNPOD_TERMINATE_AFTER}")
else:
    deleted_pod_id = POD_ID
    delete_pod(deleted_pod_id)
    POD_ID = None
    remaining = runpodctl(["pod", "list", "--all"])
    if deleted_pod_id in json.dumps(remaining):
        raise RuntimeError(
            f"Pod {deleted_pod_id} still appears in runpodctl output; verify its state"
        )
    account = runpodctl(["user"])
    print(f"Verified Pod {deleted_pod_id} is absent from the Pod list.")
    print(f"Current total spend: ${float(account.get('currentSpendPerHr', 0)):.4f}/h")
    print("Also verify the Pods, Network Volumes, and Billing pages in the console.")